# Analysis of scene classification models

This experiment evaluates the performance of pretrained scene classification models trained using the Places365 dataset under a CPU-only environment. THe objective is to analyse the trade-off between the accuaracy and the computational efficiency.

Unlike object detection models that predict bounding boxes, scene classification models assign a single category to the whole image.

The evaluation will be based on the 
-  predictive performance (top-1 and top-5 accuracy)
-  computational performance (inference latency and end-to-end processing time)

the models that will be evaluated will include
- ResNet-18 (Places365)
- ResNet-50 (Places365)
- AlexNet (Places365)

In [ ]:
# checking pytorch version and making sure it is installed
import torch
torch.__version__

'2.10.0+cpu'

## ResNet-18

The ResNet-18 model is initialized for scene classification using the Places365 dataset. Pre-trained weights are loaded from a checkpoint, with prefix adjustments applied to ensure compatibility. The model is then set to evaluation mode for inference.

In [ ]:
import torch
import torchvision.models as models
# initializing model with the 365 output classes for the places365 dataset
model = models.resnet18(num_classes=365)
# load the pretrained file
checkpoint = torch.load("resnet18_places365.pth.tar", map_location="cpu")
# extracting the model weights
state_dict = checkpoint["state_dict"]

# create a new state dictionary without the "module" prefix
# this is to ensure compatibility with the current model
new_state_dict = {}
for k, v in state_dict.items():
    new_key = k.replace("module.", "")
    new_state_dict[new_key] = v
# load the cleaned weights into the model
model.load_state_dict(new_state_dict)
model.eval()

print("Model loaded successfully")

Model loaded successfully


Before passing images into the scene classification mode, they will be preprocessed to match the format used during training. This will ensure consistent and reliable predictions.

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    # resizing, converting image to tensor and normalizing to fit the training conditions
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


The scene category labels for the Places365 dataset are loaded from a text file and processed into a clean list of class names. These labels are later used to interpret the model’s predictions by mapping output indices to human-readable scene categories.

In [ ]:
classes = []
with open("categories_places365.txt") as f:
    # extracting the class name from each line
    # removing prefix chracters and keep only the labels
    for line in f:
        classes.append(line.strip().split(' ')[0][3:])
# ensuring that the number of classes is still 365
print("Total classes:", len(classes))

Total classes: 365


A sample image is passed through the model to verify that it is functioning correctly and producing valid predictions.

In [ ]:
from PIL import Image
import torch
# loading a sample image
img = Image.open("archive/val_256/test_samples/Places365_val_00000028.jpg").convert("RGB")
# apply preprocessing and add batch dimension
input_tensor = transform(img).unsqueeze(0)

model.eval()
# run inference
with torch.no_grad():
    output = model(input_tensor)
# get predicted class index(highest score)
_, pred = torch.max(output, 1)
# map predicted index to class label and print result
print("Predicted class:", classes[pred.item()])

Predicted class: lecture_room


A quick warmup of the model is done

In [33]:
# Warmup
with torch.no_grad():
    for _ in range(10):
        model(input_tensor)

A custom dataset class is defined to load and manage the Places365 validation dataset. It reads image paths and labels from a file, filters out missing images, and applies preprocessing before returning image-label pairs for model evaluation.

In [ ]:
class Places365ValDataset(Dataset):
    # initializing dataset with image directory, label file
    def __init__(self, image_root, label_file, transform=None):
        self.image_root = image_root
        self.transform = transform
        self.samples = []
        # reading label file and store valid image paths with labels
        with open(label_file, "r") as f:
            for line in f:
                # extract relative path and class label
                rel_path, cls = line.strip().split()
                # getting file name from path
                filename = os.path.basename(rel_path)

                full_path = os.path.join(self.image_root, filename)

                if os.path.exists(full_path):  # Only keep existing images
                    self.samples.append((filename, int(cls)))
    # returning total number of samples
    def __len__(self):
        return len(self.samples)
    # retrieve a single sample (image and label) by index
    def __getitem__(self, idx):
        filename, label = self.samples[idx]
        # contrucct full image path
        img_path = os.path.join(self.image_root, filename)
        image = Image.open(img_path).convert("RGB")
        # apply preprocessing if provided
        if self.transform:
            image = self.transform(image)

        return image, label

The validation dataset is initialized and wrapped in a DataLoader to enable efficient batch processing during evaluation. The DataLoader handles batching and sequential access to the dataset without shuffling, ensuring consistent evaluation.

In [ ]:
# initialize dataset
dataset = Places365ValDataset(
    # folder containing validation images
    image_root="archive/val_256/test_samples",
    # file with image labels
    label_file="places365_val.txt",
    transform=transform
)
# create dataloader to iterate through dataset in batches
loader = DataLoader(dataset, batch_size=16, shuffle=False)

print("Total images:", len(dataset))

Total images: 7302


The model is evaluated on the validation dataset by computing Top-1 and Top-5 accuracy across all samples. Predictions are generated in batches, and performance metrics are accumulated while measuring total evaluation time.

In [ ]:
import time
import torch
import pandas as pd

top1_correct = 0
top5_correct = 0
total = 0

# start timing evaluation
start_time = time.time()

with torch.no_grad():
    for images, labels in loader:
        # foward pass through the model
        outputs = model(images)

        # compute top 1 predictions
        _, preds = torch.max(outputs, 1)
        top1_correct += (preds == labels).sum().item()

        # compute top 5 predictions
        _, top5_preds = torch.topk(outputs, 5, dim=1)
        # check if true label is within top 5 predictions
        for i in range(labels.size(0)):
            if labels[i] in top5_preds[i]:
                top5_correct += 1
        # update total number of samples processed
        total += labels.size(0)

end_time = time.time()

Evaluation metrics is calculated and the results is place inside the dataframe

In [ ]:
top1_acc = top1_correct / total
top5_acc = top5_correct / total
# calculate the total inference time per image
total_time = end_time - start_time
avg_inference_time = total_time / total

print("Top-1 Accuracy:", top1_acc)
print("Top-5 Accuracy:", top5_acc)
print("Total time:", total_time)
print("Avg inference time per image:", avg_inference_time)

Top-1 Accuracy: 0.5362914270062996
Top-5 Accuracy: 0.8359353601752945
Total time: 243.21702527999878
Avg inference time per image: 0.03330827516844683


In [54]:
results = pd.DataFrame([{
    "Model": "ResNet18",
    "Top1_Accuracy": top1_acc,
    "Top5_Accuracy": top5_acc,
    "Total_Time_sec": total_time,
    "Avg_Inference_Time_sec": avg_inference_time
}])

results

,Model,Top1_Accuracy,Top5_Accuracy,Total_Time_sec,Avg_Inference_Time_sec
0,ResNet18,0.536291,0.835935,243.217025,0.033308


## ResNet-50

the ResNet-50 model is initialised next to be tested

In [ ]:
import torch
import torchvision.models as models

# initialize resnet model for 365 scene classes
model = models.resnet50(num_classes=365)

# loading the pretrained checkpoint
checkpoint = torch.load("resnet50_places365.pth.tar", map_location="cpu")
# extracting state dictionary containing the model weights
state_dict = checkpoint["state_dict"]

# Remove "module" prefix
new_state_dict = {}
for k, v in state_dict.items():
    new_key = k.replace("module.", "")
    new_state_dict[new_key] = v

# load processed weights into model
model.load_state_dict(new_state_dict)
model.eval()

print("ResNet50 loaded successfully")

ResNet50 loaded successfully


Warming up the model

In [56]:
# Warmup
with torch.no_grad():
    for _ in range(10):
        model(input_tensor)

The model is evaluated on the validation dataset by computing Top-1 and Top-5 accuracy across all samples. Predictions are generated in batches, and performance metrics are accumulated while measuring total evaluation time.

In [ ]:
import time
import torch
import pandas as pd

top1_correct = 0
top5_correct = 0
total = 0
# start timing evaluation
start_time = time.time()

with torch.no_grad():
    for images, labels in loader:
        # foward pass through resnet model
        outputs = model(images)

        # compute top 1 predictions
        _, preds = torch.max(outputs, 1)
        top1_correct += (preds == labels).sum().item()

        # compute top 5 predictions
        _, top5_preds = torch.topk(outputs, 5, dim=1)
        # check if labels is within top 5 predictions
        for i in range(labels.size(0)):
            if labels[i] in top5_preds[i]:
                top5_correct += 1
        # update total number of samples processed
        total += labels.size(0)

end_time = time.time()

Evaluation matrics are calculated and the results is appended to the dataframe

In [58]:
top1_acc = top1_correct / total
top5_acc = top5_correct / total

total_time = end_time - start_time
avg_inference_time = total_time / total

In [59]:
results = pd.concat([
    results,
    pd.DataFrame([{
        "Model": "ResNet50",
        "Top1_Accuracy": top1_acc,
        "Top5_Accuracy": top5_acc,
        "Total_Time_sec": total_time,
        "Avg_Inference_Time_sec": avg_inference_time
    }])
], ignore_index=True)

In [60]:
results

,Model,Top1_Accuracy,Top5_Accuracy,Total_Time_sec,Avg_Inference_Time_sec
0,ResNet18,0.536291,0.835935,243.217025,0.033308
1,ResNet50,0.548891,0.848261,342.967986,0.046969


## AlexNet

The AlexNet model will then be initialised and loaded for testing

In [ ]:
import torch
import torchvision.models as models

# Create AlexNet architecture with the number of classes corresponding to places365
model = models.alexnet(num_classes=365)

# loading pretrained checkpoint
checkpoint = torch.load("alexnet_places365.pth.tar", map_location="cpu")
# extract state dictionary containing model weights
state_dict = checkpoint["state_dict"]

# Remove "module" prefix
new_state_dict = {}
for k, v in state_dict.items():
    new_state_dict[k.replace("module.", "")] = v

# load processed weights into model
model.load_state_dict(new_state_dict)
model.eval()

print("AlexNet loaded successfully")

AlexNet loaded successfully


Warmup using dummy inferences

In [62]:
# Warmup
with torch.no_grad():
    for _ in range(10):
        model(input_tensor)

The model is evaluated on the validation dataset by computing Top-1 and Top-5 accuracy across all samples. Predictions are generated in batches, and performance metrics are accumulated while measuring total evaluation time.

In [ ]:
import time
import torch
import pandas as pd

top1_correct = 0
top5_correct = 0
total = 0

# start timing evaluation
start_time = time.time()

with torch.no_grad():
    for images, labels in loader:
        # forward pass through resnet model
        outputs = model(images)

        # compute top 1 predictions
        _, preds = torch.max(outputs, 1)
        top1_correct += (preds == labels).sum().item()

        # compute top 5 predictions
        _, top5_preds = torch.topk(outputs, 5, dim=1)

        # check if true labels is within top 5 predictions
        for i in range(labels.size(0)):
            if labels[i] in top5_preds[i]:
                top5_correct += 1
        # update total number of samples processed
        total += labels.size(0)

end_time = time.time()

Evaluation matrics are calculated and the results is appended to the dataframe

In [64]:
top1_acc = top1_correct / total
top5_acc = top5_correct / total

total_time = end_time - start_time
avg_inference_time = total_time / total

In [65]:
results = pd.concat([
    results,
    pd.DataFrame([{
        "Model": "AlexNet",
        "Top1_Accuracy": top1_acc,
        "Top5_Accuracy": top5_acc,
        "Total_Time_sec": total_time,
        "Avg_Inference_Time_sec": avg_inference_time
    }])
], ignore_index=True)

In [66]:
results

,Model,Top1_Accuracy,Top5_Accuracy,Total_Time_sec,Avg_Inference_Time_sec
0,ResNet18,0.536291,0.835935,243.217025,0.033308
1,ResNet50,0.548891,0.848261,342.967986,0.046969
2,AlexNet,0.471241,0.772391,76.647088,0.010497


ResNet18 was selected because it provides the best balance between accuracy and computational efficiency for this task. Although ResNet50 achieves slightly higher Top-1 and Top-5 accuracy, the improvement is marginal while requiring noticeably longer inference time. In contrast, AlexNet offers the fastest inference speed, but its accuracy is substantially lower, resulting in a meaningful drop in classification reliability. ResNet18 achieves near-ResNet50 performance while reducing average inference time to approximately 0.033 seconds per image. In practical terms, analysing a single image takes only around 33 milliseconds, which is sufficiently fast for most real-world applications while maintaining strong predictive performance. Therefore, ResNet18 represents the most balanced and efficient choice for this use case.

Weights obtained from: https://github.com/CSAILVision/places365

Places365 Dataset: https://www.kaggle.com/datasets/pankajkumar2002/places365

Validation txt file: filelist_places365-standard.tar

